# Ice Tea Sales Analysis and Forecasting

This project analyses 52 weeks of ice tea sales to understand what drives revenue: advertising (Search, Newspaper, Social Media), pricing, temperature, and holidays. The goal is to build a regression model that can forecast revenue and support marketing budget decisions.

The regression framework follows James et al., *An Introduction to Statistical Learning with Python* (2023, Chapter 3).



## Table of Contents

**A. Data Import and Quality Checks**
- A.1 Libraries and Data Loading
- A.2 Variable Types
- A.3 Data Quality

**B. Exploratory Data Analysis (EDA)**
- B.1 Revenue Over Time
- B.2 Revenue and Temperature
- B.3 Advertising Budget and Timing
- B.4 Scatter Plots
- B.5 Correlation Analysis

**C. Multicollinearity Diagnostics (VIF)**

**D. Ordinary Least Squares Modeling (OLS)**
- D.1 Baseline vs Full Model
- D.2 Coefficient Interpretation

**E. Train/Test Forecasting**

**F. Residual Analysis**

**G. Conclusions and Recommendations**

## A. Data Import and Quality Checks

### A.1 Libraries and Data Loading

In [4]:
#import libraries

import pandas as pd     #data manipulation
import numpy as np      #numerical operations
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm         #OLS regression and inference
from statsmodels.stats.outliers_influence import variance_inflation_factor as VIF   #multicollinearity diagnostic
from statsmodels.stats.stattools import durbin_watson

from sklearn.metrics import mean_squared_error as MSE
from scipy import stats

In [5]:
#mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
#load dataset
df = pd.read_csv('/content/drive/MyDrive/GitHub/icetea_weekly_sales.csv')

In [ ]:
#check dimensions, column names, data types and null values
df.info()

The dataset has 52 weekly observations and 7 columns. Revenue is the dependent variable, the other six are predictors.

### A.2 Variable Types

I check how many distinct values each variable takes to decide how to treat them in the model.

In [ ]:
#statistical summary: Revenue is Y, the others are X
df.describe().round(2)

In [ ]:
#Newspaper only takes 2 values, so it is binary
df['Newspaper'].value_counts()

In [ ]:
#same for Social Media
df['Social_media'].value_counts()

In [ ]:
#Price delta takes only 3 values, so I treat it as categorical
df['Price_delta'].value_counts()

In [ ]:
#Holiday is binary by definition
df['Holiday'].value_counts()

In [ ]:
#Search takes many values, so I treat it as continuous
df['Search'].value_counts()

Newspaper, Social Media, and Holiday are binary. Price Delta only takes three fixed values so I treat it as categorical. Search and Temperature vary continuously and are treated as continuous. Revenue is a continuous monetary variable.

### A.3 Data Quality

In [ ]:
#check missing values
df.isnull().sum()

In [ ]:
#check duplicates
df.duplicated().sum()

No missing values, no duplicates. The dataset is clean.

## B. Exploratory Data Analysis (EDA)

### B.1 Revenue Over Time

I plot revenue across 52 weeks to get a first look at the seasonal pattern and see whether holiday weeks stand out.

In [ ]:
#revenue over time with holiday weeks highlighted
plt.figure(figsize=(12, 5))
plt.plot(df['Week'], df['Revenue'], color='darkblue', linewidth=1.5, marker='o', markersize=4)

#highlight holiday weeks to see if they behave differently
holidays = df[df['Holiday'] == 1]
plt.scatter(holidays['Week'], holidays['Revenue'], color='red', s=80, zorder=5, label='Holiday weeks')

plt.xlabel('Week')
plt.ylabel('Revenue ($)')
plt.title('Weekly Ice Tea Revenue Over 52 Weeks')
plt.legend()
plt.tight_layout()
plt.show()

Revenue is low in winter, rises through spring, peaks in summer (weeks 24-35, max 5428 in week 28) and drops again in autumn. Holiday weeks consistently spike above their neighbours throughout the whole year, not just in summer. Week 8 in winter reached 3251 while surrounding weeks stayed around 2600, which suggests Holiday has an effect independent from seasonality.

### B.2 Revenue and Temperature

Ice tea is a warm-weather product, so temperature is the most obvious seasonal driver. I overlay the two series to see how closely they move together.

In [ ]:
#temperature over time to see the seasonal shape
plt.figure(figsize=(12, 5))
plt.plot(df['Week'], df['Temperature'], color='orange', linewidth=1.5, marker='o', markersize=4)
plt.xlabel('Week')
plt.ylabel('Temperature (°F)')
plt.title('Weekly Temperature Over 52 Weeks')
plt.tight_layout()
plt.show()

In [ ]:
#overlay revenue and temperature on the same chart to compare their shapes
fig, ax1 = plt.subplots(figsize=(12, 5))

ax1.plot(df['Week'], df['Revenue'], color='darkblue', linewidth=1.5, marker='o', markersize=4, label='Revenue')
holidays = df[df['Holiday'] == 1]
ax1.scatter(holidays['Week'], holidays['Revenue'], color='red', s=80, zorder=5, label='Holiday weeks')
ax1.set_xlabel('Week')
ax1.set_ylabel('Revenue (euros)')

#second y-axis for temperature since the scales are different
ax2 = ax1.twinx()
ax2.plot(df['Week'], df['Temperature'], color='orange', linewidth=1.5, linestyle='--', label='Temperature')
ax2.set_ylabel('Temperature (°F)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Revenue and Temperature Over 52 Weeks')
plt.tight_layout()
plt.show()

The two curves are almost identical in shape. Peak revenue in week 28 coincides with peak temperature (77F), and the lowest revenue week is also one of the coldest. Week 28 is also a holiday week though, so multiple factors overlap here and visual inspection alone cannot separate them.

### B.3 Advertising Budget and Timing

I look at how the budget is split across channels and when it was actually spent during the year.

In [ ]:
#total spend per channel to see budget allocation
plt.figure(figsize=(8, 5))
plt.bar('Newspaper', df['Newspaper'].sum(), label='Newspaper', alpha=0.5)
plt.bar('Search', df['Search'].sum(), label='Search', alpha=0.5)
plt.bar('Social Media', df['Social_media'].sum(), label='Social Media', alpha=0.5)
plt.xlabel('Channel')
plt.ylabel('Total Spend ($)')
plt.title('Total Advertising Budget Allocation')
plt.legend()
plt.show()

Newspaper received the largest share of total spend. Total spend alone does not tell us much about timing though, so I look at the weekly distribution.

In [ ]:
#weekly spend by channel stacked to see timing
plt.figure(figsize=(12, 5))
plt.bar(df['Week'], df['Newspaper'], label='Newspaper', alpha=0.5)
plt.bar(df['Week'], df['Search'], bottom=df['Newspaper'], label='Search', alpha=0.5)
plt.bar(df['Week'], df['Social_media'], bottom=df['Newspaper'] + df['Search'], label='Social Media', alpha=0.5)
plt.xlabel('Week')
plt.ylabel('Spend (euros)')
plt.title('Weekly Advertising Spend by Channel')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#overlay spend and revenue to see if they move together
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.bar(df['Week'], df['Newspaper'], label='Newspaper', alpha=0.5)
ax1.bar(df['Week'], df['Search'], bottom=df['Newspaper'], label='Search', alpha=0.5)
ax1.bar(df['Week'], df['Social_media'], bottom=df['Newspaper'] + df['Search'], label='Social Media', alpha=0.5)
ax1.set_xlabel('Week')
ax1.set_ylabel('Advertising Spend (euros)')

ax2 = ax1.twinx()
ax2.plot(df['Week'], df['Revenue'], color='darkblue', linewidth=2, marker='o', markersize=4, label='Revenue')
ax2.set_ylabel('Revenue (euros)')

holidays = df[df['Holiday'] == 1]
plt.scatter(holidays['Week'], holidays['Revenue'], color='red', s=80, zorder=5, label='Holiday weeks')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Advertising Spend and Revenue Over Time')
plt.tight_layout()
plt.show()

All three channels are active only during weeks 24-35. Advertising, temperature, and revenue all peak at exactly the same time, which is going to make it hard to separate the advertising effect from seasonal demand in the regression.

### B.4 Scatter Plots

Scatter plots remove the time dimension and show the direct relationship between each predictor and revenue.

In [ ]:
#scatter plots for all predictors to check individual relationships with revenue
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].scatter(df['Temperature'], df['Revenue'], alpha=0.6)
axes[0, 0].set_xlabel('Temperature (°F)')
axes[0, 0].set_ylabel('Revenue (euros)')
axes[0, 0].set_title('Revenue vs Temperature')

axes[0, 1].scatter(df['Search'], df['Revenue'], alpha=0.6)
axes[0, 1].set_xlabel('Search Spend (euros)')
axes[0, 1].set_ylabel('Revenue (euros)')
axes[0, 1].set_title('Revenue vs Search')

axes[1, 0].scatter(df['Newspaper'], df['Revenue'], alpha=0.6)
axes[1, 0].set_xlabel('Newspaper Spend (euros)')
axes[1, 0].set_ylabel('Revenue (euros)')
axes[1, 0].set_title('Revenue vs Newspaper')

axes[1, 1].scatter(df['Social_media'], df['Revenue'], alpha=0.6)
axes[1, 1].set_xlabel('Social Media Spend (euros)')
axes[1, 1].set_ylabel('Revenue (euros)')
axes[1, 1].set_title('Revenue vs Social Media')

plt.tight_layout()
plt.show()

In [ ]:
#zoom in on temperature since it shows the strongest relationship
plt.figure(figsize=(8, 5))
sns.scatterplot(x='Temperature', y='Revenue', data=df)
plt.title('Revenue vs Temperature')
plt.xlabel('Temperature (°F)')
plt.ylabel('Revenue (euros)')
plt.show()

Temperature shows the clearest linear relationship with revenue. Newspaper shows a binary split between campaign and non-campaign weeks, but since all campaigns ran in summer the temperature effect is mixed in. Search and Social Media have the same problem: non-zero values only appear in warm weeks, so their individual contribution cannot be read from scatter plots alone.

### B.5 Correlation Analysis

Pearson correlation gives a numerical measure of how strongly variables move together. I focus on inter-predictor correlations too, not just with revenue, because high correlations between predictors are what cause multicollinearity.

In [ ]:
#Pearson correlation matrix — I round to 3 decimals for readability
correlation_matrix = df.corr().round(3)
correlation_matrix

In [ ]:
#heatmap to visualize correlations between all variables at once
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix')
plt.show()

Revenue correlates most with Newspaper (0.85), Temperature (0.82), and Search (0.82). More importantly, Search and Newspaper are correlated at 0.90 and Search and Temperature at 0.88. Advertising and temperature moved together all year, which is exactly the problem that will show up in the regression.

## C. Multicollinearity Diagnostics (VIF)

VIF measures how much each predictor overlaps with the others. Below 5 is acceptable, 5-10 is moderate concern, above 10 is severe.

In [ ]:
#compute VIF for each predictor
predictors = ['Newspaper', 'Search', 'Social_media', 'Price_delta', 'Temperature', 'Holiday']
X = sm.add_constant(df[predictors])
vif_data = pd.DataFrame({'Variable': predictors,
'VIF': [VIF(X.values, i+1) for i in range(len(predictors))]})
#sort descending to see the most problematic variables first
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)
vif_data.round(2)

In [ ]:
#color code by severity: red > 10, orange 5-10, green < 5
plt.figure(figsize=(10, 5))

colors = ['#D32F2F' if v > 10 else '#FF9800' if v > 5 else '#4CAF50' for v in vif_data['VIF']]
plt.barh(vif_data['Variable'], vif_data['VIF'], color=colors)
plt.axvline(x=5, color='orange', linestyle='--', label='Moderate (VIF=5)')
plt.axvline(x=10, color='red', linestyle='--', label='Severe (VIF=10)')
plt.xlabel('VIF')
plt.title('Variance Inflation Factor by Predictor')
plt.legend()
plt.tight_layout()
plt.show()

Search has a VIF of 15.47, the only variable in the severe zone. It overlaps too much with Temperature and Newspaper for the model to estimate its coefficient reliably. Newspaper (6.71) and Temperature (6.24) are in the moderate zone. The other three predictors are fine.

I keep Search in the model because it is the main continuous advertising channel and removing it would make the results less useful. Multicollinearity does not affect the model's overall fit or forecasts, only the precision of individual coefficients.

## D. Ordinary Least Squares Modeling (OLS)

### D.1 Baseline vs Full Model

I start with a simple baseline using only Newspaper, the predictor with the highest correlation with revenue (r = 0.85), to have a reference point before adding all six predictors.

In [ ]:
#baseline: Revenue ~ Newspaper only
X_baseline = sm.add_constant(df['Newspaper'])
baseline_model = sm.OLS(df['Revenue'], X_baseline).fit()
print(baseline_model.summary())

The baseline explains 72% of revenue variance (R2 = 0.72). The Newspaper coefficient is 8.67, so a full campaign week (200 euros) is associated with roughly 1734 euros in extra revenue. This is inflated though, because Newspaper only ran in summer and the model is picking up part of the temperature effect.

In [ ]:
#full model: Revenue ~ all predictors
predictors = ['Newspaper', 'Search', 'Social_media', 'Price_delta', 'Temperature', 'Holiday']
X = sm.add_constant(df[predictors])
model = sm.OLS(df['Revenue'], X).fit()
print(model.summary())

The full model reaches R2 = 0.97. Five out of six predictors are significant. Search is the only exception (p = 0.62), as expected from the VIF results.

The Newspaper coefficient dropped from 8.67 to 4.50 once Temperature is in the model. The baseline was attributing part of summer demand to Newspaper when temperature was the actual driver. A 200 euros campaign is now estimated at around 900 euros in extra revenue.

### D.2 Coefficient Interpretation

Each coefficient represents the estimated revenue change for a one-unit increase in that predictor, holding all others constant.

Holiday has the largest effect at 804 euros per week. Since this shows up throughout the year and not just in summer, it is genuinely independent from the seasonal pattern.

Temperature adds 27.61 euros per degree Fahrenheit. Across the full seasonal range in the dataset (32F to 80F), that accounts for roughly 1325 euros in revenue variation.

Newspaper contributes 4.50 euros per euro spent. Social Media contributes 3.71 euros per euro spent and is significant (p = 0.0001) despite only being active in summer.

Price Delta has a coefficient of -752 euros per unit, so a price increase of 0.5 is associated with a drop of around 376 euros in revenue.

Search is not significant (p = 0.62). This does not mean Search has no effect, it means the dataset cannot measure it separately from temperature because the two always moved together.

I export the dataset with the model's predicted values and residuals to **Excel**, together with the coefficients. The cleaned and formatted version of this file is available on **GitHub**.

## E. Train/Test Forecasting

An R2 of 0.97 on training data does not guarantee the model generalises. I split the data chronologically into weeks 1-41 for training and weeks 42-52 for testing. I use a chronological split rather than a random one because the model should predict future weeks from past data, not the other way around.

In [ ]:
#chronological split: 80% train, 20% test
#I use a chronological split because this is time series data
split = 41
train = df.iloc[:split]
test = df.iloc[split:]
predictors = ['Newspaper', 'Search', 'Social_media', 'Price_delta', 'Temperature', 'Holiday']
X_train = sm.add_constant(train[predictors])
X_test = sm.add_constant(test[predictors])
y_train = train['Revenue']
y_test = test['Revenue']
print(f'Train: {len(train)} weeks (1-41)')
print(f'Test: {len(test)} weeks (42-52)')

Weeks 42-52 cover mid-autumn to year-end: temperatures drop from 61F to 33F, Newspaper and Social Media are inactive, Search is minimal. The test period is quite different from the training data, which makes this a meaningful check.

In [ ]:
#fit on training data only
train_model = sm.OLS(y_train, X_train).fit()
print(train_model.summary())

Coefficients on the training set are very close to the full model in Section D. Search is again non-significant (p = 0.18). The model is stable.

In [ ]:
#predict on test set
#add constant to align dimensions with the training model
X_test_fixed = sm.add_constant(test[predictors], has_constant='add')
y_pred = train_model.predict(X_test_fixed)

#evaluation metrics
rmse_test = np.sqrt(MSE(y_test, y_pred))
r2_test = 1 - np.sum((y_test - y_pred)**2) / np.sum((y_test - y_test.mean())**2)
mae_test = np.mean(np.abs(y_test - y_pred))
mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

print(f'R2 (test): {r2_test:.4f}')
print(f'RMSE: {rmse_test:.2f}')
print(f'MAE: {mae_test:.2f}')
print(f'MAPE: {mape:.2f}%')

Test MAPE is 5%, so predictions are off by 5% on average on unseen data. The best prediction is week 48, a holiday week: actual 3856 euros, predicted 3790 euros. The worst is week 46 with no holiday and no advertising, where something outside the model drove revenue up unexpectedly.

In [ ]:
#plot actual vs predicted for the full timeline
plt.figure(figsize=(14, 6))

plt.plot(train['Week'], y_train, color='darkblue', marker='o', markersize=4,
linewidth=1.5, label='Actual (train)')
plt.plot(train['Week'], train_model.predict(X_train), color='steelblue',
linestyle='--', linewidth=1.5, label='Predicted (train)')

plt.plot(test['Week'], y_test, color='darkblue', marker='o', markersize=4, linewidth=1.5)
plt.plot(test['Week'], y_pred, color='red', linestyle='--', marker='s',
markersize=5, linewidth=1.5, label='Predicted (test)')

#vertical line to mark where training ends and test begins
plt.axvline(x=41.5, color='gray', linestyle=':', label='Train/Test split')

plt.xlabel('Week')
plt.ylabel('Revenue (euros)')
plt.title('Actual vs Predicted Revenue')
plt.legend()
plt.tight_layout()
plt.show()

The test predictions follow the autumn downward trend correctly with no systematic pattern of over or under-prediction.

## F. Residual Analysis

OLS relies on a few key assumptions. If they are violated, the coefficients and p-values from Section D are not reliable. I check the two most important ones visually.

Linearity: residuals should be randomly scattered around zero with no pattern. A curve or funnel shape would suggest the linear model is missing something.

In [ ]:
#extract fitted values and residuals from the full model
fitted = model.fittedvalues
residuals = model.resid

#residuals vs fitted: if linearity holds, points should be randomly scattered around zero
plt.figure(figsize=(10, 6))
plt.scatter(fitted, residuals, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--')
plt.xlabel('Fitted Values (euros)')
plt.ylabel('Residuals (euros)')
plt.title('Residuals vs Fitted Values')
plt.tight_layout()
plt.show()

Residuals are randomly scattered around zero with constant spread. No pattern visible. Linearity holds.

Normality: p-values and confidence intervals from OLS assume residuals follow a normal distribution. I check with a Q-Q plot and a histogram.

In [ ]:
#Q-Q plot to check normality visually and histogram to see the distribution shape
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stats.probplot(residuals, dist='norm', plot=axes[0])
axes[0].set_title('Q-Q Plot of Residuals')

axes[1].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Residual (euros)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')

plt.tight_layout()
plt.show()

Residuals fall closely along the Q-Q diagonal and the histogram is roughly symmetric. Normality holds. The results in Section D can be trusted.

## G. Conclusions and Recommendations

The full model explains 97% of revenue variance and achieves a test MAPE of 5% on unseen data. The five significant drivers are Holiday (804 euros per week), Temperature (27.61 euros per degree F), Newspaper (4.50 euros per euro spent), Social Media (3.71 euros per euro spent), and Price Delta (-752 euros per unit). Search could not be evaluated reliably because it always ran alongside high temperatures.

The most actionable finding is around holiday weeks. The model estimates an organic spike of 804 euros on holiday weeks, but 8 out of 10 holiday weeks outside summer currently have zero advertising spend. Adding a 200 euros Newspaper campaign to those 8 weeks could generate around 7200 euros in additional annual revenue. The Holiday coefficient is stable and confirmed on the test set, so this is the recommendation I am most confident about.

A second opportunity is spreading Newspaper campaigns across the year instead of concentrating all 12 in weeks 24-35. Summer demand is already high because of temperature, so the marginal return on advertising in July is probably lower than in spring or autumn. Spreading campaigns would also generate the data variation needed to measure advertising effects independently from temperature in future seasons.

For Search, the only way to measure its real contribution is to run a campaign when temperatures are low and the other channels are inactive. If revenue rises beyond what temperature alone would predict, that confirms it works. If not, around 1020 euros per year could be reallocated to channels with proven returns.

Finally, with a 5% MAPE the model is reliable enough to use for budget planning. The team can input planned spend, expected temperature and known holidays to project weekly revenue before committing to the next season.